<a href="https://colab.research.google.com/github/LunaTic-Neon/2026-1-NLP/blob/main/26_1_0327_NLP_w4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import re
!pip install konlpy
from konlpy.tag import Okt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. 실제 데이터 로드
df = pd.read_csv('/content/sample_data/daum_movie_review.csv')

# 결측치 제거
df = df.dropna(subset=['review', 'title'])

# 독립변수(X)와 종속변수(y) 설정
X = df['review']
y = df['title']

# 2. Train / Test 데이터 분할 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"학습 데이터 크기: {X_train.shape}, 테스트 데이터 크기: {X_test.shape}")

학습 데이터 크기: (11780,), 테스트 데이터 크기: (2945,)


In [5]:
okt = Okt()

def custom_tokenizer(text):

    # 1. 형태소/품사 결합
    pos_words = [f"{word}/{tag}" for word, tag in okt.pos(text)]

    # 2. 길이가 5 이상인 토큰만 추출
    filtered_words = [word for word in pos_words if len(word) >= 5]

    return filtered_words

# 3. TF-IDF Vectorizer 설정
tfidf_vect = TfidfVectorizer(tokenizer=custom_tokenizer, max_features=None, token_pattern=None)

# 학습 데이터 및 테스트 데이터 변환
print("TF-IDF 변환 중...")
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

print(f"추출된 TF-IDF 특성(Feature) 개수: {len(tfidf_vect.get_feature_names_out())}")

TF-IDF 변환 중...
추출된 TF-IDF 특성(Feature) 개수: 25503


In [6]:
results = {}

# 1. Logistic Regression
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(X_train_tfidf, y_train)
results['1. Logistic Regression'] = accuracy_score(y_test, lr_clf.predict(X_test_tfidf))

# 2. Ridge Classifier
ridge_params = {'alpha': [0.1, 1.0, 5.0, 10.0]}
ridge_grid = GridSearchCV(RidgeClassifier(random_state=42), param_grid=ridge_params, cv=3, n_jobs=-1)
ridge_grid.fit(X_train_tfidf, y_train)
best_ridge = ridge_grid.best_estimator_
results[f'2. Ridge (alpha={ridge_grid.best_params_["alpha"]})'] = accuracy_score(y_test, best_ridge.predict(X_test_tfidf))

# 3. Lasso 정규화
lasso_clf = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42)
lasso_clf.fit(X_train_tfidf, y_train)
results['3. Lasso (L1 Logistic)'] = accuracy_score(y_test, lasso_clf.predict(X_test_tfidf))

# 4. MultinomialNB
nb_clf = MultinomialNB(alpha=1.0)
nb_clf.fit(X_train_tfidf, y_train)
results['4. Multinomial NB'] = accuracy_score(y_test, nb_clf.predict(X_test_tfidf))

print("기본 4개 모델 학습 완료.")

기본 4개 모델 학습 완료.


In [7]:
ngram_ranges = {'Unigram': (1, 1), 'Bigram': (1, 2), 'Trigram': (1, 3)}

for name, ngram in ngram_ranges.items():
    print(f"[{name}] 변환 및 학습 중...")
    ngram_vect = TfidfVectorizer(tokenizer=custom_tokenizer, max_features=None, token_pattern=None, ngram_range=ngram)

    # N-gram 변환
    X_train_ngram = ngram_vect.fit_transform(X_train)
    X_test_ngram = ngram_vect.transform(X_test)

    # Logistic Regression 학습
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_ngram, y_train)

    # 정확도 기록
    acc = accuracy_score(y_test, clf.predict(X_test_ngram))
    results[f'{list(ngram_ranges.keys()).index(name) + 5}. N-gram ({name})'] = acc

print("N-gram 모델 3종 학습 완료.")

[Unigram] 변환 및 학습 중...
[Bigram] 변환 및 학습 중...
[Trigram] 변환 및 학습 중...
N-gram 모델 3종 학습 완료.


In [9]:
print("\n" + "-"*43)
print("[ 7개 영화 제목 분류기 테스트 정확도 비교 ]")
print("-"*43)

# 정확도 내림차순 정렬
sorted_results = sorted(results.items(), key=lambda item: item[1], reverse=True)

for idx, (model_name, acc) in enumerate(sorted_results, 1):
    # 정확도를 소수점 넷째 자리까지
    print(f"{idx}. {model_name:<28}: {acc:.4f}")
print("-"*43)


-------------------------------------------
[ 7개 영화 제목 분류기 테스트 정확도 비교 ]
-------------------------------------------
1. 2. Ridge (alpha=1.0)        : 0.7175
2. 1. Logistic Regression      : 0.6971
3. 5. N-gram (Unigram)         : 0.6971
4. 3. Lasso (L1 Logistic)      : 0.6890
5. 6. N-gram (Bigram)          : 0.6764
6. 7. N-gram (Trigram)         : 0.6564
7. 4. Multinomial NB           : 0.4781
-------------------------------------------
